In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

print("=" * 70)
print("PROCESSED E-COMMERCE DATASET - DAY 9")
print("=" * 70)

# Load the three datasets
orders = pd.read_csv("Day9_Orders.csv")
customers = pd.read_csv("Day9_Customers.csv")
products = pd.read_csv("Day9_Products.csv")

print("\n1. ORDERS DATASET")
print("Shape:", orders.shape)
display(orders.head())

print("\n2. CUSTOMERS DATASET")
print("Shape:", customers.shape)
display(customers.head())

print("\n3. PRODUCTS DATASET")
print("Shape:", products.shape)
display(products.head())

print("\n4. COLUMN NAMES")
print("Orders:", orders.columns.tolist())
print("Customers:", customers.columns.tolist())
print("Products:", products.columns.tolist())

print("\n5. DATA TYPES")
print("\nOrders:")
print(orders.dtypes)
print("\nCustomers:")
print(customers.dtypes)
print("\nProducts:")
print(products.dtypes)

print("\n6. MISSING VALUES")
print("\nOrders:")
print(orders.isnull().sum())
print("\nCustomers:")
print(customers.isnull().sum())
print("\nProducts:")
print(products.isnull().sum())

# Convert order date to datetime
date_column = None

for col in orders.columns:
    if col.lower().replace("_", "").replace(" ", "") in [
        "orderdate", "date", "order_date"
    ]:
        date_column = col
        break

if date_column is None:
    for col in orders.columns:
        try:
            converted = pd.to_datetime(orders[col])
            if converted.notna().sum() > len(orders) * 0.7:
                date_column = col
                break
        except:
            pass

if date_column:
    orders[date_column] = pd.to_datetime(orders[date_column], errors="coerce")
else:
    print("\nWarning: Order date column could not be identified.")

# Identify merge columns
customer_key = None
product_key = None

for col in customers.columns:
    if col.lower().replace("_", "").replace(" ", "") == "customerid":
        customer_key = col
        break

for col in products.columns:
    if col.lower().replace("_", "").replace(" ", "") == "productid":
        product_key = col
        break

order_customer_key = None
order_product_key = None

for col in orders.columns:
    clean_col = col.lower().replace("_", "").replace(" ", "")
    if clean_col == "customerid":
        order_customer_key = col
    if clean_col == "productid":
        order_product_key = col

print("\n7. IDENTIFIED MERGE KEYS")
print("Customer Key:", customer_key)
print("Order Customer Key:", order_customer_key)
print("Product Key:", product_key)
print("Order Product Key:", order_product_key)

# Merge Orders and Customers
if customer_key and order_customer_key:
    processed = pd.merge(
        orders,
        customers,
        left_on=order_customer_key,
        right_on=customer_key,
        how="left"
    )
else:
    processed = orders.copy()

print("\n8. AFTER MERGING ORDERS AND CUSTOMERS")
print("Shape:", processed.shape)
display(processed.head())

# Merge Products
if product_key and order_product_key:
    processed = pd.merge(
        processed,
        products,
        left_on=order_product_key,
        right_on=product_key,
        how="left",
        suffixes=("", "_Product")
    )

print("\n9. AFTER MERGING PRODUCTS")
print("Shape:", processed.shape)
display(processed.head())

# Demonstrate concat()
orders_sample_1 = orders.head(5)
orders_sample_2 = orders.tail(5)

concatenated_orders = pd.concat(
    [orders_sample_1, orders_sample_2],
    ignore_index=True
)

print("\n10. CONCAT() DEMONSTRATION")
print("Combined first 5 and last 5 order records:")
display(concatenated_orders)

# DateTime operations
if date_column and date_column in processed.columns:
    processed["Order_Year"] = processed[date_column].dt.year
    processed["Order_Month"] = processed[date_column].dt.month
    processed["Order_Month_Name"] = processed[date_column].dt.month_name()
    processed["Order_Day"] = processed[date_column].dt.day
    processed["Order_Day_Name"] = processed[date_column].dt.day_name()
    processed["Order_Week"] = processed[date_column].dt.isocalendar().week.astype(int)

    print("\n11. DATETIME OPERATIONS")
    display(
        processed[
            [date_column, "Order_Year", "Order_Month",
             "Order_Month_Name", "Order_Day", "Order_Day_Name",
             "Order_Week"]
        ].head(10)
    )

# Use apply() to create useful columns
numeric_columns = processed.select_dtypes(include=np.number).columns.tolist()

if "Quantity" in processed.columns:
    processed["Quantity_Level"] = processed["Quantity"].apply(
        lambda x: "High" if x >= 3 else "Low"
    )

if "Rating" in processed.columns:
    processed["Rating_Level"] = processed["Rating"].apply(
        lambda x: "Excellent" if x >= 4.5
        else ("Good" if x >= 3.5 else "Average")
    )

# Create Total Sales if possible
if "Quantity" in processed.columns:
    price_column = None

    for col in processed.columns:
        clean_col = col.lower().replace("_", "").replace(" ", "")
        if clean_col in ["unitprice", "price", "productprice"]:
            price_column = col
            break

    if price_column and "Total_Sales" not in processed.columns:
        processed["Total_Sales"] = processed["Quantity"] * processed[price_column]

# Apply() for customer membership
if "Membership_Type" in processed.columns:
    processed["Customer_Type"] = processed["Membership_Type"].apply(
        lambda x: "Premium Customer" if str(x).lower() == "premium"
        else "Regular Customer"
    )

print("\n12. APPLY() OPERATIONS")
display(processed.head(10))

# Remove duplicate columns
processed = processed.loc[:, ~processed.columns.duplicated()]

# Remove duplicate rows
processed = processed.drop_duplicates()

# Clean string columns
for col in processed.select_dtypes(include="object").columns:
    processed[col] = processed[col].astype(str).str.strip()

# Reorganize columns
preferred_columns = [
    "Order_ID",
    "Order_Date",
    "Order_Year",
    "Order_Month",
    "Order_Month_Name",
    "Order_Day",
    "Order_Day_Name",
    "Customer_ID",
    "Customer_Name",
    "City",
    "Region",
    "Membership_Type",
    "Customer_Type",
    "Product_ID",
    "Product_Name",
    "Category",
    "Quantity",
    "Unit_Price",
    "Total_Sales",
    "Rating",
    "Payment_Method",
    "Quantity_Level",
    "Rating_Level"
]

existing_preferred = [
    col for col in preferred_columns if col in processed.columns
]

remaining_columns = [
    col for col in processed.columns
    if col not in existing_preferred
]

processed = processed[existing_preferred + remaining_columns]

print("\n13. FINAL PROCESSED DATASET")
print("Rows:", processed.shape[0])
print("Columns:", processed.shape[1])
display(processed.head(10))

print("\n14. FINAL DATASET INFORMATION")
processed.info()

print("\n15. FINAL MISSING VALUES")
display(processed.isnull().sum())

# Save processed dataset
output_file = "Day9_Processed_Ecommerce_Dataset.csv"
processed.to_csv(output_file, index=False)

print("\n" + "=" * 70)
print("PROCESSING COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Final dataset saved as:", output_file)
print("Final shape:", processed.shape)
print("File location:", os.path.abspath(output_file))

# Download the processed file in Google Colab
try:
    from google.colab import files
    files.download(output_file)
except:
    print("If running outside Google Colab, the file is saved in the current folder.")

PROCESSED E-COMMERCE DATASET - DAY 9

1. ORDERS DATASET
Shape: (120, 7)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered



2. CUSTOMERS DATASET
Shape: (30, 5)


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New



3. PRODUCTS DATASET
Shape: (20, 5)


...


14. FINAL DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Order_ID          120 non-null    object        
 1   Order_Date        120 non-null    datetime64[ns]
 2   Order_Year        120 non-null    int32         
 3   Order_Month       120 non-null    int32         
 4   Order_Month_Name  120 non-null    object        
 5   Order_Day         120 non-null    int32         
 6   Order_Day_Name    120 non-null    object        
 7   Customer_ID       120 non-null    object        
 8   Customer_Name     120 non-null    object        
 9   City              120 non-null    object        
 10  Region            120 non-null    object        
 11  Membership_Type   120 non-null    object        
 12  Customer_Type     120 non-null    object        
 13  Product_ID        120 non-null    object        


,0
Order_ID,0
Order_Date,0
Order_Year,0
Order_Month,0
Order_Month_Name,0
Order_Day,0
Order_Day_Name,0
Customer_ID,0
Customer_Name,0
City,0



PROCESSING COMPLETED SUCCESSFULLY
Final dataset saved as: Day9_Processed_Ecommerce_Dataset.csv
Final shape: (120, 24)
File location: /content/Day9_Processed_Ecommerce_Dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>